# 📐 Evaluation als Spezifikation

> *"In klassischer Software definieren Tests die Korrektheit. In KI-Software definieren **Metriken** die Korrektheit."*

Deine Metrik-Funktion IST deine formale Spezifikation. Sie sagt dem System, was "richtig" bedeutet. Ohne Metrik bist du blind.

Der Weg ist klar: **Bau dir eine Metrik → teste sie mit Fake-Daten → lass das echte Modell laufen → sieh die Scores.** Von "Ich weiss nicht ob's gut ist" zu "Ich kann exakt messen wie gut."

| Klassische Software | KI-Software |
|---|---|
| Unit Test | Metrik-Funktion |
| `assert x == y` | `metric(expected, predicted) → score` |
| Pass/Fail (binär) | 0.0 bis 1.0 (kontinuierlich) |
| Testet Code | Testet Modell-Output |

In [ ]:
import sys; sys.path.insert(0, ".")
import dspy
import ipywidgets as widgets
from IPython.display import display
from dspy_tasks.tasks import get_task, list_by_tier
from dspy_tasks.calculations import (METRIC_REGISTRY, code_execution_proxy, analogy_match,
                                      fact_verdict_accuracy, numeric_match)
from dspy_tasks.actions import run_baseline, _evaluate_examples, _mean
from dspy_tasks.visualize import *
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

MODELS = get_available_models()
model_dd = model_picker(MODELS, default=get_default_model())
display(model_dd)

In [ ]:
from dspy_tasks.visualize import diagram_compare

diagram_compare(
    {"title": "Klassische Software", "items": ["Unit Test", "assert x == y", "Pass/Fail"], "icon": "🔧", "color": "#8a8886"},
    {"title": "KI-Software", "items": ["Metrik-Funktion", "metric(gold, pred) → score", "0.0 bis 1.0"], "icon": "🧠", "color": "#0078d4"},
    title="Tests vs. Metriken"
)

## Metriken werden immer raffinierter

Von simplem Exact-Match über Token-F1 bis hin zu gewichteten Composite-Scores — je besser deine Metrik, desto präziser kannst du optimieren und vergleichen.

In [ ]:
# Level 1: Binary exact match (simplest)
print("Level 1: Exact Match")
print(f"  'positive' == 'positive' → {1.0}")
print(f"  'positive' == 'POSITIVE' → {1.0}  (normalized)")
print(f"  'positive' == 'negative' → {0.0}")

# Level 2: Token F1 (partial credit)
from dspy_tasks.calculations import token_f1
print(f"\nLevel 2: Token F1")
print(f"  gold=['apple','banana'], pred=['apple','cherry'] → {token_f1(['apple','banana'], ['apple','cherry']):.3f}")

# Level 3: Composite weighted
print(f"\nLevel 3: Composite (ticket routing)")
print("  Priority correct (40%) + Category correct (35%) + Team correct (25%)")
print("  = Weighted specification of 'what matters most'")

display_insight("Der Kern-Insight",
    "Deine Metrik-Funktion IST deine Produkt-Spezifikation. "
    "Wenn du die Gewichte änderst, änderst du, wofür das System optimiert. "
    "Deshalb ist 'Evaluation die Software der Zukunft'.")

In [ ]:
from dspy_tasks.config import configure_dspy
# Code Generation task — different metric than simple matching
task = get_task("code_generation")
print(f"Task: {task.name}")
print(f"Metric: code_execution_proxy (checks structure, keywords, overlap)")
print(f"Teaching point: {task.teaching_point}\n")

btn = run_button("Evaluate Code Generation")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        configure_dspy(model=model_dd.value)
        result = run_baseline("code_generation", model_dd.value, max_eval=8)
        display_score("Code Generation", result.score)
        display_results_table(result.individual_scores)

btn.on_click(on_run)
display(widgets.HBox([model_dd, btn]), out)

In [ ]:
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in [get_task(tid) for tid in ["code_generation", "analogy", "fact_verification"]]],
    description="Task:")
compare_btn = run_button("Compare All Models")
compare_out = widgets.Output()

def on_compare(b):
    with compare_out:
        compare_out.clear_output()
        print(f"⏳ Evaluating {task_dd.value} across {len(MODELS)} models...")
        scores = {}
        for m in MODELS:
            result = run_baseline(task_dd.value, m, max_eval=8)
            scores[m] = {"baseline": result.score}
            display_score(m.split("/")[-1], result.score)

        fig = bar_comparison(get_task(task_dd.value).name, scores)
        fig.show()

compare_btn.on_click(on_compare)
display(widgets.HBox([task_dd, compare_btn]), compare_out)

## Verschiedene Metriken, verschiedene Rankings

Das Gleiche Output kann unter verschiedenen Metriken unterschiedlich gut abschneiden. **Deine Metrik zu wählen heisst, deine Werte zu wählen.**

In [ ]:
display_insight("Evaluation = Spezifikation",
    "In klassischer Software schreibst du Tests NACH dem Code. "
    "In KI-Software schreibst du Metriken VOR der Optimierung. "
    "Die Metrik IST die Spezifikation. Der Optimizer findet Code (Prompts), der deine Tests besteht.",
    icon="📐")

## ⏭️ Weiter geht's!

Du hast Metriken. Du kannst messen. Aber was, wenn der **Computer die Prompts SELBST optimieren** könnte?

Stell dir vor: du schreibst die Spezifikation (Signatur + Metrik), und ein Optimizer findet den besten Prompt dafür. Genau wie ein Compiler! Das ist das nächste Notebook.